# Uncertainty Estimation

Estimate tree-level prediction uncertainty and evaluate interval calibration. This notebook rebuilds the required features from the processed FD001 files; it does not depend on another notebook being run first.

In [ ]:
from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
RUL_CAP = 125

np.random.seed(RANDOM_STATE)

print("FleetGuard Phase 2 initialized.")

In [ ]:
# Colab upload option:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    PROJECT_ROOT = Path("/content")
    DATA_DIR = PROJECT_ROOT
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "data" / "processed").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_csv(DATA_DIR / "train_fd001.csv")
val = pd.read_csv(DATA_DIR / "validation_fd001.csv")
test = pd.read_csv(DATA_DIR / "test_fd001_with_rul.csv")

print("Training:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

assert set(train.engine_id).isdisjoint(set(val.engine_id))
print("Engine leakage check: PASSED")


## 1. Feature engineering
Raw sensor readings are useful, but their recent trends can provide additional information about engine degradation.

We'll use:
- Current sensor values.
- Rolling mean over 5, 10 and 20 cycles.
- Rolling standard deviation over 5, 10 and 20 cycles.
- Current operating settings.
- Current operating cycle.

### Select useful sensors

In [ ]:
sensor_cols = [
    f"sensor_{i:02d}" for i in range(1, 22)
]

setting_cols = [
    f"setting_{i}" for i in range(1, 4)
]

constant_sensors = [
    "sensor_01",
    "sensor_05",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

sensor_cols = [
    c for c in sensor_cols
    if c not in constant_sensors
]

print("Selected sensors:", len(sensor_cols))
print(sensor_cols)

### Create rolling features

In [ ]:
def create_features(df):

    df = df.copy()

    df = df.sort_values(
        ["engine_id", "cycle"]
    ).reset_index(drop=True)

    feature_frames = [df]

    for window in [5, 10, 20]:

        rolling_mean = (
            df.groupby("engine_id")[sensor_cols]
            .rolling(window, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        rolling_mean.columns = [
            f"{c}_mean_{window}" for c in sensor_cols
        ]

        rolling_std = (
            df.groupby("engine_id")[sensor_cols]
            .rolling(window, min_periods=2)
            .std()
            .reset_index(level=0, drop=True)
        )

        rolling_std.columns = [
            f"{c}_std_{window}" for c in sensor_cols
        ]

        feature_frames.extend([
            rolling_mean,
            rolling_std
        ])

    result = pd.concat(feature_frames, axis=1)

    result = result.fillna(0)

    return result

### Generate training features

In [ ]:
train_features = create_features(train)

print("Training feature shape:", train_features.shape)

display(train_features.head())

### Generate truncated validation trajectories
We will simulate the situation where an engine is still operating and only its history up to a particular cycle is available.

For each validation engine, we select a cutoff corresponding to an actual remaining life between 5 and 125 cycles.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

val_truncated = []

for engine_id, group in val.groupby("engine_id"):

    group = group.sort_values("cycle")

    max_cycle = group["cycle"].max()

    max_remaining = min(RUL_CAP, max_cycle - 1)

    actual_remaining = rng.integers(
        5, max_remaining + 1
    )

    cutoff = max_cycle - actual_remaining

    partial_history = group[
        group["cycle"] <= cutoff
    ].copy()

    val_truncated.append(partial_history)

val_truncated = pd.concat(
    val_truncated,
    ignore_index=True
)

print("Original validation rows:", len(val))
print("Truncated validation rows:", len(val_truncated))
print("Validation engines:", val_truncated.engine_id.nunique())

### Extract the final observation per validation engine
The model will predict RUL using the latest available observation for each engine.

In [ ]:
val_features = create_features(val_truncated)

val_points = (
    val_features
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .reset_index(drop=True)
)

print("Validation prediction points:", len(val_points))

display(
    val_points[
        ["engine_id", "cycle", "rul_uncapped", "rul"]
    ].head()
)

###Prepare model inputs

In [ ]:
DROP_COLS = [
    "engine_id",
    "rul",
    "rul_uncapped",
    "rul_at_last_cycle"
]

feature_cols = [
    c for c in train_features.columns
    if c not in DROP_COLS
]

X_train = train_features[feature_cols]
y_train = train_features["rul"]

X_val = val_points[feature_cols]
y_val = val_points["rul_uncapped"]

print("Training X:", X_train.shape)
print("Validation X:", X_val.shape)

In [ ]:
test_features = create_features(test)
test_points = (
    test_features
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)
X_test = test_points[feature_cols]
print("Test prediction points:", X_test.shape)


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"
FIGURE_DIR = OUTPUT_DIR / "figures"
for folder in [MODEL_DIR, RESULT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


Extract per-tree predictions and compute uncertainty

In [ ]:
# Train on the 80-engine training partition only (leakage-safe).
from sklearn.ensemble import RandomForestRegressor

N_TREES = 300
uncertainty_rf = RandomForestRegressor(
    n_estimators=N_TREES,
    max_depth=15,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
uncertainty_rf.fit(X_train, y_train)
print(f"Uncertainty ensemble trained: {N_TREES} trees on {X_train.shape[0]} rows, {X_train.shape[1]} features")
print(f"Training engines: {train.engine_id.nunique()} (disjoint from validation)")


In [ ]:
def get_tree_predictions(rf, X):
    """Return array of shape (n_samples, n_trees) with one prediction per tree."""
    preds = np.array([tree.predict(X) for tree in rf.estimators_])
    return preds.T  # (n_samples, n_trees)


def compute_uncertainty_estimates(tree_preds, rul_cap=RUL_CAP):
    """
    From per-tree predictions, compute:
      - mean (point estimate)
      - std (epistemic uncertainty)
      - p5, p95 (90% prediction interval)
    Individual tree predictions are clipped to [0, RUL_CAP] before aggregation.
    """
    clipped = np.clip(tree_preds, 0, rul_cap)
    return {
        "pred_mean": clipped.mean(axis=1),
        "pred_std": clipped.std(axis=1),
        "p5": np.percentile(clipped, 5, axis=1),
        "p95": np.percentile(clipped, 95, axis=1),
        "p10": np.percentile(clipped, 10, axis=1),
        "p90": np.percentile(clipped, 90, axis=1),
    }


# Validation set (20 engines — ground truth available for calibration)
val_tree_preds = get_tree_predictions(uncertainty_rf, X_val)
val_unc = compute_uncertainty_estimates(val_tree_preds)

# Test set (100 engines — ground truth available for evaluation)
test_tree_preds = get_tree_predictions(uncertainty_rf, X_test)
test_unc = compute_uncertainty_estimates(test_tree_preds)

# Assemble dataframes
val_uncertainty_df = pd.DataFrame({
    "engine_id": val_points["engine_id"].values,
    "actual_rul": y_val.values,
    "pred_mean": val_unc["pred_mean"],
    "pred_std": val_unc["pred_std"],
    "ci90_lower": val_unc["p5"],
    "ci90_upper": val_unc["p95"],
    "ci80_lower": val_unc["p10"],
    "ci80_upper": val_unc["p90"],
})
val_uncertainty_df["ci90_width"] = val_uncertainty_df["ci90_upper"] - val_uncertainty_df["ci90_lower"]
val_uncertainty_df["abs_error"] = np.abs(val_uncertainty_df["pred_mean"] - val_uncertainty_df["actual_rul"])

test_uncertainty_df = pd.DataFrame({
    "engine_id": test_points["engine_id"].values,
    "actual_rul": test_points["rul_uncapped"].values,
    "pred_mean": test_unc["pred_mean"],
    "pred_std": test_unc["pred_std"],
    "ci90_lower": test_unc["p5"],
    "ci90_upper": test_unc["p95"],
    "ci80_lower": test_unc["p10"],
    "ci80_upper": test_unc["p90"],
})
test_uncertainty_df["ci90_width"] = test_uncertainty_df["ci90_upper"] - test_uncertainty_df["ci90_lower"]
test_uncertainty_df["abs_error"] = np.abs(test_uncertainty_df["pred_mean"] - test_uncertainty_df["actual_rul"])

print("Validation uncertainty shape:", val_uncertainty_df.shape)
print("Test uncertainty shape:", test_uncertainty_df.shape)
print("\n--- Validation (first 10 engines) ---")
display(val_uncertainty_df[["engine_id", "actual_rul", "pred_mean", "pred_std",
                             "ci90_lower", "ci90_upper"]].round(2).head(10))
print("\n--- Test (first 10 engines) ---")
display(test_uncertainty_df[["engine_id", "actual_rul", "pred_mean", "pred_std",
                              "ci90_lower", "ci90_upper"]].round(2).head(10))

Evaluate uncertainty calibration on validation set

In [ ]:
def evaluate_interval_coverage(df, lower_col, upper_col, actual_col="actual_rul"):
    """Compute empirical coverage and average interval width."""
    actual = df[actual_col].values
    lower = df[lower_col].values
    upper = df[upper_col].values
    inside = (actual >= lower) & (actual <= upper)
    return {
        "empirical_coverage": inside.mean(),
        "avg_width": (upper - lower).mean(),
        "miss_below": int((actual < lower).sum()),
        "miss_above": int((actual > upper).sum()),
        "n": len(df),
    }

cal_90 = evaluate_interval_coverage(val_uncertainty_df, "ci90_lower", "ci90_upper")
cal_80 = evaluate_interval_coverage(val_uncertainty_df, "ci80_lower", "ci80_upper")

print("=" * 60)
print("UNCERTAINTY CALIBRATION — Validation Set (20 engines)")
print("=" * 60)
print(f"{'Interval':<12} {'Nominal':<10} {'Empirical':<12} {'Avg Width':<12} {'Miss<':<8} {'Miss>':<8}")
print("-" * 60)
print(f"{'90%':<12} {'90.0%':<10} {cal_90['empirical_coverage']:.1%}      {cal_90['avg_width']:.2f}       {cal_90['miss_below']:<8} {cal_90['miss_above']:<8}")
print(f"{'80%':<12} {'80.0%':<10} {cal_80['empirical_coverage']:.1%}      {cal_80['avg_width']:.2f}       {cal_80['miss_below']:<8} {cal_80['miss_above']:<8}")

# Uncertainty-error correlation (are uncertain predictions actually less accurate?)
corr_pearson = val_uncertainty_df["pred_std"].corr(val_uncertainty_df["abs_error"], method="pearson")
corr_spearman = val_uncertainty_df["pred_std"].corr(val_uncertainty_df["abs_error"], method="spearman")

print(f"\n--- Uncertainty ↔ Error Correlation ---")
print(f"Pearson:  {corr_pearson:.3f}")
print(f"Spearman: {corr_spearman:.3f}")

# Split by median uncertainty and compare error magnitudes
median_std = val_uncertainty_df["pred_std"].median()
high = val_uncertainty_df[val_uncertainty_df["pred_std"] >= median_std]
low = val_uncertainty_df[val_uncertainty_df["pred_std"] < median_std]

print(f"\n--- Error by Uncertainty Group (median split, std={median_std:.2f}) ---")
print(f"High-uncertainty engines:  n={len(high):2d}  mean abs error = {high['abs_error'].mean():.2f} cycles")
print(f"Low-uncertainty engines:   n={len(low):2d}  mean abs error = {low['abs_error'].mean():.2f} cycles")
print(f"\nInterpretation: {'Uncertainty IS informative (higher std → larger error).' if corr_pearson > 0 else 'WARNING: uncertainty NOT correlated with error.'}")

Identify high-uncertainty test predictions

In [ ]:
# Define high-uncertainty as top quartile of prediction std
std_q75 = test_uncertainty_df["pred_std"].quantile(0.75)
std_q50 = test_uncertainty_df["pred_std"].quantile(0.50)

test_uncertainty_df["uncertainty_tier"] = pd.cut(
    test_uncertainty_df["pred_std"],
    bins=[-np.inf, std_q50, std_q75, np.inf],
    labels=["low", "medium", "high"]
)

# Error breakdown by tier
tier_summary = (
    test_uncertainty_df
    .groupby("uncertainty_tier", observed=True)
    .agg(
        n_engines=("engine_id", "count"),
        mean_abs_error=("abs_error", "mean"),
        mean_std=("pred_std", "mean"),
        mean_ci90_width=("ci90_width", "mean"),
    )
    .round(2)
)

print("=" * 65)
print(f"HIGH-UNCERTAINTY TEST PREDICTIONS (top quartile, std >= {std_q75:.2f})")
print("=" * 65)
print(f"\nError breakdown by uncertainty tier:")
display(tier_summary)

# Show the highest-uncertainty engines
high_unc_engines = (
    test_uncertainty_df[test_uncertainty_df["uncertainty_tier"] == "high"]
    .sort_values("pred_std", ascending=False)
)
print(f"\n--- {len(high_unc_engines)} high-uncertainty engines (sorted by std) ---")
display(high_unc_engines[["engine_id", "actual_rul", "pred_mean", "pred_std",
                           "ci90_lower", "ci90_upper", "ci90_width", "abs_error"]].round(2))

Visualize prediction uncertainty

In [ ]:
# Figure 1: Prediction intervals vs actual RUL (validation + test)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df, title in [
    (axes[0], val_uncertainty_df, "Validation (20 engines)"),
    (axes[1], test_uncertainty_df, "Test (100 engines)"),
]:
    sorted_df = df.sort_values("pred_mean").reset_index(drop=True)
    x = np.arange(len(sorted_df))
    ax.vlines(x, sorted_df["ci90_lower"], sorted_df["ci90_upper"],
              color="steelblue", alpha=0.6, label="90% prediction interval")
    ax.scatter(x, sorted_df["pred_mean"], color="steelblue", alpha=0.8,
               s=20, label="Predicted RUL")
    ax.scatter(x, sorted_df["actual_rul"], color="crimson", marker="x",
               s=40, zorder=5, label="Actual RUL")
    ax.set_xlabel("Engine (sorted by predicted RUL)")
    ax.set_ylabel("RUL (cycles)")
    ax.set_title(f"{title}: Prediction Intervals")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase1_prediction_intervals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: phase1_prediction_intervals.png")


In [ ]:
# Figure 2: Uncertainty vs error + predicted-vs-actual colored by uncertainty
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Validation: uncertainty vs absolute error
ax = axes[0]
ax.scatter(val_uncertainty_df["pred_std"], val_uncertainty_df["abs_error"],
           alpha=0.7, s=60, color='steelblue', edgecolors='navy', linewidth=0.5)
if len(val_uncertainty_df) > 2:
    z = np.polyfit(val_uncertainty_df["pred_std"], val_uncertainty_df["abs_error"], 1)
    p = np.poly1d(z)
    xs = np.linspace(val_uncertainty_df["pred_std"].min(), val_uncertainty_df["pred_std"].max(), 100)
    ax.plot(xs, p(xs), 'r--', alpha=0.8, label=f"Linear fit (r={corr_pearson:.2f})")
ax.set_xlabel("Prediction Std Dev (uncertainty)")
ax.set_ylabel("Absolute Error (cycles)")
ax.set_title("Validation: Uncertainty vs Prediction Error")
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Test: predicted vs actual, colored by uncertainty
ax = axes[1]
sc = ax.scatter(test_uncertainty_df["actual_rul"], test_uncertainty_df["pred_mean"],
                c=test_uncertainty_df["pred_std"], cmap='YlOrRd',
                s=50, alpha=0.8, edgecolors='black', linewidth=0.4)
ax.plot([0, RUL_CAP], [0, RUL_CAP], 'k--', alpha=0.4, label="Perfect prediction")
plt.colorbar(sc, ax=ax, label="Prediction Std (uncertainty)")
ax.set_xlabel("Actual RUL")
ax.set_ylabel("Predicted RUL")
ax.set_title("Test: Predicted vs Actual (color = uncertainty)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase1_uncertainty_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: phase1_uncertainty_analysis.png")

Save outputs for downstream phases

In [ ]:
# Save dataframes
val_uncertainty_df.to_csv(RESULT_DIR / "validation_uncertainty.csv", index=False)
test_uncertainty_df.to_csv(RESULT_DIR / "test_uncertainty.csv", index=False)

# Save calibration metrics
calibration_df = pd.DataFrame([
    {"interval": "90%", "nominal_coverage": 0.90,
     "empirical_coverage": cal_90["empirical_coverage"],
     "avg_width": cal_90["avg_width"],
     "miss_below": cal_90["miss_below"], "miss_above": cal_90["miss_above"],
     "n_engines": cal_90["n"]},
    {"interval": "80%", "nominal_coverage": 0.80,
     "empirical_coverage": cal_80["empirical_coverage"],
     "avg_width": cal_80["avg_width"],
     "miss_below": cal_80["miss_below"], "miss_above": cal_80["miss_above"],
     "n_engines": cal_80["n"]},
])
calibration_df.to_csv(RESULT_DIR / "uncertainty_calibration.csv", index=False)

# Save the uncertainty ensemble model
joblib.dump(uncertainty_rf, MODEL_DIR / "uncertainty_ensemble.pkl")

print("Phase 1 outputs saved:")
for f in ["validation_uncertainty.csv", "test_uncertainty.csv",
          "uncertainty_calibration.csv", "uncertainty_ensemble.pkl"]:
    print(f"  outputs/results or outputs/models / {f}")

print("\n" + "=" * 60)
print("PHASE 1 SUMMARY: Uncertainty Estimation")
print("=" * 60)
print(f"Method: Random Forest tree-level ensemble ({N_TREES} trees)")
print(f"  → Each tree = 1 sample in the prediction distribution")
print(f"  → std across trees = epistemic uncertainty")
print(f"  → 5th/95th percentiles = 90% prediction interval")
print(f"\nCalibration (validation, 20 engines):")
print(f"  90% interval: nominal 90% → empirical {cal_90['empirical_coverage']:.0%} | width {cal_90['avg_width']:.1f} cycles")
print(f"  80% interval: nominal 80% → empirical {cal_80['empirical_coverage']:.0%} | width {cal_80['avg_width']:.1f} cycles")
print(f"\nUncertainty informativeness:")
print(f"  Pearson (std ↔ |error|):  {corr_pearson:.3f}")
print(f"  Spearman (std ↔ |error|): {corr_spearman:.3f}")
print(f"  High-unc group mean |error|: {high['abs_error'].mean():.1f} vs Low-unc: {low['abs_error'].mean():.1f}")
print(f"\nTest set: {len(high_unc_engines)} high-uncertainty engines (top quartile)")
print(f"\n→ test_uncertainty_df ready for Phase 2 (Risk Assessment)")
print(f"→ val_uncertainty_df ready for calibration reference")